# 🛵 Projeto Final — Análise Exploratória de Dados

## RangoJá: o que faz uma entrega demorar?

**Módulo:** Análise Exploratória de Dados
**Conteúdo avaliado:** Aulas 1 a 4 (EDA, qualidade e limpeza, estatística, visualização e storytelling)
**Tempo estimado:** 2 a 3 horas

## 📬 Instruções de entrega

| | |
|---|---|
| **O que enviar** | **Este mesmo notebook**, com todas as células de resposta preenchidas (código e textos) |
| **Para onde** | Por e-mail, para **jchambyd@gmail.com** |
| **Prazo** | Até **20/10/2026** |
| **Formato** | Individual **ou** em dupla (uma única entrega por dupla) |
| **Nome do arquivo** | `projeto_eda_<nome1>_<nome2>.ipynb` (ex.: `projeto_eda_ana_bruno.ipynb`) |
| **Assunto do e-mail** | `Projeto EDA – <Nome 1> / <Nome 2>` |

**Antes de enviar:**
1. Execute **Kernel → Restart & Run All** (no Colab: *Ambiente de execução → Reiniciar e executar tudo*) e confira que **tudo roda sem erro, do início ao fim**;
2. **Salve o notebook com as saídas** (tabelas e gráficos visíveis) — o professor vai ler o notebook sem precisar executá-lo;
3. **Não apague nem altere** os enunciados e a célula de validação da Parte 2.

### 👥 Identificação

✍️ **Preencha:**

- **Nome 1:**
- **Nome 2 (se for dupla):**

## 📖 O contexto

A **RangoJá** é um app de delivery de comida (fictício) que opera na cidade de **Vila Serena**. A gerente de operações, **Marina**, está preocupada com reclamações sobre **demora nas entregas** e pediu a sua ajuda. Ela exportou do sistema os pedidos de **março a maio de 2025** no arquivo **`pedidos_rangoja.csv`** (coloque-o na mesma pasta deste notebook).

Marina quer respostas para quatro perguntas:

| # | Pergunta da Marina | Parte do projeto |
|---|---|---|
| 1 | *"Os pedidos demoram mais em dias de chuva? Quanto?"* | Parte 3 — Estatística |
| 2 | *"A nota que os clientes dão depende do tipo de restaurante?"* | Parte 3 — Estatística |
| 3 | *"Em quais horários do dia as entregas são mais lentas?"* | Parte 4 — Visualização |
| 4 | *"Quais bairros têm as entregas mais demoradas? É só por causa da distância?"* | Parte 4 — Visualização |

Mas atenção: o arquivo veio **direto do sistema**, sem nenhum tratamento. Antes de responder qualquer coisa, você vai precisar **diagnosticar** e **limpar** os dados.

### 📖 Dicionário de dados

| Coluna | Descrição |
|---|---|
| `id_pedido` | Identificador do pedido |
| `data_hora` | Data e hora em que o pedido foi feito |
| `bairro` | Bairro de entrega (Vila Serena tem **10 bairros** atendidos — lista abaixo) |
| `distancia_km` | Distância entre o restaurante e o endereço de entrega (km) |
| `categoria` | Categoria do restaurante (Lanches, Pizza, Japonesa, Brasileira, Saudável) |
| `valor_pedido` | Valor total do pedido |
| `forma_pagamento` | Pix, Cartão ou Dinheiro |
| `cupom` | O cliente usou cupom de desconto? (Sim/Não) |
| `chovendo` | Estava chovendo na cidade no dia do pedido? (Sim/Não) |
| `tempo_prometido_min` | Tempo de entrega prometido pelo app no momento da compra (minutos) |
| `tempo_entrega_min` | Tempo **real** de entrega, do pedido até a chegada (minutos) |
| `avaliacao` | Nota dada pelo cliente, de 1 a 5 estrelas (**vazio = o cliente não avaliou**) |

**Os 10 bairros atendidos:** Centro, Vila Nova, Jardim das Flores, Santa Luzia, Bosque, Alto Alegre, Morada do Sol, Parque das Águas, Cidade Nova e Distrito Industrial.

## 🧮 Critérios de avaliação

| Parte | O que é avaliado | Pontos |
|---|---|---|
| **1. Diagnóstico** | Problemas identificados com evidência (código) e descritos com clareza | 15 |
| **2. Limpeza e qualidade** | Duplicatas + 3 colunas tratadas corretamente, com **auditoria** e **justificativa** das decisões; validação aprovada | 30 |
| **3. Estatística** | Descritivas, teste adequado e justificado, IC, tamanho do efeito, conclusão correta (2 perguntas × 15) | 30 |
| **4. Visualização** | Gráficos explicativos (tipo adequado, título-ação, faxina, cor com propósito) e interpretação (2 perguntas × 10) | 20 |
| **Qualidade geral** | Notebook roda do início ao fim; textos claros; conclusões escritas para a Marina (sem jargão) | 5 |
| **Bônus** | Pergunta extra (atraso × avaliação) | +10 |
| | **Total** | **100 (+10)** |

> 💡 **Dica geral:** o que mais pesa na nota **não** é o código mais curto ou mais sofisticado, e sim as **decisões bem justificadas** e as **conclusões corretas**. Sempre que o enunciado pedir "justifique", escreva **por que** você fez aquilo. Reaproveite à vontade as funções das aulas (`relatorio_qualidade`, `normalizar_texto`, `converter_datas`, `bootstrap_ic`, `cohen_d`, `estilo_storytelling`...) — basta copiá-las para este notebook.

---

## ⚙️ Preparação (já pronta — apenas execute)

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

bruto = pd.read_csv("pedidos_rangoja.csv")
print("Arquivo carregado:", bruto.shape)
backup = bruto

Arquivo carregado: (2660, 12)


> 📌 Regra de ouro (Aula 2): **não altere o `bruto`**. Na Parte 2 você vai trabalhar em uma cópia chamada **`df`**.

Se quiser, cole na célula abaixo as funções das aulas que pretende reutilizar:

In [ ]:
# (Opcional) Funções reaproveitadas das aulas 👇
'''
# CONFIG VISUAIS
plt.rcParams["figure.figsize"] = (10, 4)       # tamanho padrão dos gráficos (largura, altura)
plt.rcParams["axes.spines.top"] = False        # remove a borda de cima
plt.rcParams["axes.spines.right"] = False      # remove a borda da direita
plt.rcParams["axes.grid"] = True               # liga a grade
plt.rcParams["grid.alpha"] = 0.3               # grade bem clarinha

# Cores que usaremos em todos os gráficos
COR_PRINCIPAL = "#2a6fdb"   # azul
COR_DESTAQUE = "#d1495b"    # vermelho (para chamar atenção)
COR_NEUTRA = "#8a8f98"      # cinza

# Mostra no máximo 20 colunas ao exibir tabelas
pd.set_option("display.max_columns", 20)

print("Bibliotecas carregadas com sucesso! ✅")
print("Versão do pandas:", pd.__version__)

# REMOVE ESPAÇOS
def normalizar_texto(serie):
    """Remove espaços nas pontas, coloca em minúsculas e remove acentos."""
    return (serie.str.strip()
                 .str.lower()
                 .str.normalize("NFKD")
                 .str.encode("ascii", "ignore")
                 .str.decode("ascii"))


cidade_norm = normalizar_texto(df["cidade"])
print("Depois da normalização mecânica:", cidade_norm.nunique(), "valores")
cidade_norm.value_counts()


# BOOLEANOS
MAPA_SIM_NAO = {
    "sim": True, "s": True, "1": True, "yes": True,
    "nao": False, "não": False, "n": False, "0": False, "no": False,
}

pet_texto = df["aceita_pet"].str.strip().str.lower()
pet_bool = pet_texto.map(MAPA_SIM_NAO)

# Auditoria: sobrou algum texto que o mapa não conhece?
nao_mapeados = pet_texto[pet_texto.notna() & pet_bool.isna()].unique()
print("Valores não mapeados:", list(nao_mapeados))


# NULOS DISFARÇADOS

SUSPEITOS = ["-", "?", "n/a", "na", "null", "none", "sem info", "consulte", "não informado", ""]

def procurar_nulos_disfarcados(dados, suspeitos=SUSPEITOS):
    """Conta, em cada coluna, valores de texto que parecem representar 'ausência'."""
    achados = {}
    for col in dados.columns:
        texto = dados[col].dropna().astype(str).str.strip().str.lower()
        contagem = texto[texto.isin(suspeitos)].value_counts()
        if len(contagem) > 0:
            achados[col] = contagem.to_dict()
    return achados


procurar_nulos_disfarcados(df)



---

# Parte 1 — Diagnóstico dos dados (15 pontos)

📚 *Revise: Aula 1 (seção 3.5 e checklist de EDA) e Aula 2 (seção 1).*

### Tarefa 1.1 — Inspecione o arquivo

Usando o DataFrame `bruto`, mostre **com código**:
- o número de linhas e colunas;
- as primeiras linhas **e** uma amostra aleatória;
- o tipo de cada coluna e a quantidade de valores ausentes;
- a quantidade de **linhas duplicadas**;
- as estatísticas descritivas (`describe`) e a contagem de valores (`value_counts`) das colunas em que isso for útil;
- os **nulos disfarçados** (textos que representam ausência).

In [3]:
# Seu código aqui 👇
# Quantidade de linhas e colunas
print("Linhas: ", bruto.shape[0])
print("Colunas: ", bruto.shape[1])


Linhas:  2660
Colunas:  12


In [4]:
# Seu código aqui 👇
# As primeiras linhas e uma amostra aleatória
print("Primeiras linhas: ")
print(bruto.head())

print("\nAmostra aleatória: ")
print(bruto.sample(np.random.seed(42)))


Primeiras linhas: 
   id_pedido            data_hora          bairro  distancia_km categoria  \
0     501668  2025-04-27 20:16:00       Vila Nova           2.5     Pizza   
1     500687  2025-03-25 11:27:00          Bosque           3.0     Pizza   
2     500945  2025-04-02 10:58:00     Alto Alegre           4.7  Saudável   
3     502228     18/05/2025 19h46     Cidade Nova           7.5   Lanches   
4     501110  2025-04-07 21:26:00  Jd. das Flores           3.2  Saudável   

  valor_pedido forma_pagamento cupom chovendo  tempo_prometido_min  \
0     R$ 70,29          Cartão   Não      Não                   35   
1     R$ 72,86        Dinheiro   Não      Sim                   30   
2     R$ 42,47          Cartão   Não      Não                   35   
3     R$ 61,34          Cartão   Não      Não                   55   
4     R$ 29,22             pix   Não      Não                   30   

  tempo_entrega_min  avaliacao  
0                43        4.0  
1                32        5.0 

In [5]:
# Seu código aqui 👇
# O tipo de cada coluna e a quantidade de valores ausentes
print("Tipos de cada coluna:\n", bruto.dtypes)

print("\n")

valores_nulos = bruto.isna().sum()

print("Quantidade de valores ausentes: \n", valores_nulos.sort_values(ascending=False))



#print("\n")

#print("Porcentagem de ausentes: ", bruto.isna().sum() / len(bruto) * 100)

Tipos de cada coluna:
 id_pedido                int64
data_hora               object
bairro                  object
distancia_km           float64
categoria               object
valor_pedido            object
forma_pagamento         object
cupom                   object
chovendo                object
tempo_prometido_min      int64
tempo_entrega_min       object
avaliacao              float64
dtype: object


Quantidade de valores ausentes: 
 avaliacao              794
tempo_entrega_min       45
distancia_km            40
id_pedido                0
data_hora                0
bairro                   0
categoria                0
valor_pedido             0
forma_pagamento          0
cupom                    0
chovendo                 0
tempo_prometido_min      0
dtype: int64


In [6]:
# Seu código aqui 👇
# Quantidades de linhas duplicadas
linhas_duplicadas = bruto[bruto.duplicated(keep=False)]
print("Linhas duplicadas: \n", linhas_duplicadas)
print(f"Total de linhas duplicadas: {bruto.duplicated().sum()}\n")

Linhas duplicadas: 
       id_pedido            data_hora             bairro  distancia_km  \
12       500031     02/03/2025 12:07      Jd das Flores           2.8   
13       500056  2025-03-02 20:59:00   Parque das Águas           5.6   
19       500096     03/03/2025 21:35  Jardim das Flores           2.9   
25       500260     09/03/2025 21h19   Parque das Águas           7.5   
31       500088  2025-03-03 19:48:00        Santa Luzia           4.9   
...         ...                  ...                ...           ...   
2476     502156     15/05/2025 20:38             Bosque           3.6   
2509     501175     10/04/2025 18:21        ALTO ALEGRE           6.5   
2530     500696  2025-03-25 12:31:00        Santa Luzia           3.7   
2533     501533     23/04/2025 11h37      Morada do Sol           4.8   
2632     500611     22/03/2025 18h16             Bosque           2.9   

       categoria valor_pedido forma_pagamento cupom chovendo  \
12      Japonesa     R$ 72,28         

In [7]:
# Seu código aqui 👇
# Estatísticas descritivas e contagem de valores
print("Análise descritiva: \n", bruto.describe())

print("\n")

# Categorias de Comida mais pedidas
print("Categorias mais Pedidas:")
print(bruto['categoria'].value_counts(ascending=False))

# Bairros com maior volume de pedidos
print("\nVolume de Pedidos por Bairro:")
print(bruto['bairro'].value_counts(ascending=False).head(10))

# Formas de Pagamento mais utilizadas
print("\nFormas de Pagamento:")
print(bruto['forma_pagamento'].value_counts(ascending=False))

# Uso de Cupom de Desconto
print("\nClientes que usaram Cupom:")
print(bruto['cupom'].value_counts(ascending=False))

# Impacto do Clima (Chuva)
print("\nPedidos feitos na Chuva:")
print(bruto['chovendo'].value_counts(ascending=False))

# Distribuição das Notas/Avaliações
print("\nDistribuição de Avaliações")
print(bruto['avaliacao'].value_counts(ascending=False))

#for col in bruto.select_dtypes(include=['object', 'category']).columns:
#    print(f"\nContagem para a coluna [{col}]:")
#    print(bruto[col].value_counts().head(10))
# Não acho que seja necessário o value_counts no momento

Análise descritiva: 
            id_pedido  distancia_km  tempo_prometido_min    avaliacao
count    2660.000000   2620.000000          2660.000000  1866.000000
mean   501301.910526      3.890191            37.321429     4.414255
std       752.458171      2.092931             8.519389     0.612988
min    500001.000000      0.000000            20.000000     2.000000
25%    500650.750000      2.300000            30.000000     4.000000
50%    501300.500000      3.500000            35.000000     4.000000
75%    501954.250000      5.300000            40.000000     5.000000
max    502600.000000      9.700000            65.000000     5.000000


Categorias mais Pedidas:
categoria
Lanches       796
Pizza         589
Brasileira    576
Saudável      373
Japonesa      326
Name: count, dtype: int64

Volume de Pedidos por Bairro:
bairro
Centro                 366
Vila Nova              308
Jardim das Flores      265
Santa Luzia            229
Bosque                 198
Alto Alegre            186
Mora

In [8]:
# Seu código aqui 👇
# Nulos disfarçados
SUSPEITOS = ["-", "?", "n/a", "na", "null", "none", "sem info", "consulte", "não informado", "", "NULL", "NA", "N/A", "NONE"]

def procurar_nulos_disfarcados(dados, suspeitos=SUSPEITOS):
    """Conta, em cada coluna, valores de texto que parecem representar 'ausência'."""
    achados = {}
    for col in dados.columns:
        texto = dados[col].dropna().astype(str).str.strip().str.lower()
        contagem = texto[texto.isin(suspeitos)].value_counts()
        if len(contagem) > 0:
            achados[col] = contagem.to_dict()
    return achados

procurar_nulos_disfarcados(bruto)



{'tempo_entrega_min': {'?': 8}}

In [9]:

print(f"Total de bairros diferentes: {bruto['bairro'].nunique()}")
print(f"Total de formas de pagamento diferentes: {bruto['forma_pagamento'].nunique()}")

print("Bairros únicos cadastrados:")
print(bruto['bairro'].unique())

print("\nFormas de pagamento únicas cadastradas:")
print(bruto['forma_pagamento'].unique())



Total de bairros diferentes: 43
Total de formas de pagamento diferentes: 9
Bairros únicos cadastrados:
['Vila Nova' 'Bosque' 'Alto Alegre' 'Cidade Nova' 'Jd. das Flores'
 'Distrito Industrial' 'Vila nova' 'Santa Luzia' 'bosque' 'Centro'
 'Jd das Flores' 'Parque das Águas' 'Morada do Sol' 'morada do sol'
 'Jardim das Flores' 'Distr. Industrial' 'Centro ' 'centro' 'Santa luzia '
 'CENTRO' 'Alto alegre' 'BOSQUE' 'Pq. das Águas' 'Parque das Aguas'
 'Morada Do Sol' ' Bosque' 'V. Nova' 'Morada do sol '
 'distrito industrial' 'Jardim Das Flores' 'parque das águas' 'vila nova'
 'VILA NOVA' 'PARQUE DAS AGUAS' 'Distrito industrial' 'CIDADE NOVA'
 'santa luzia' 'alto alegre' 'Sta. Luzia' 'ALTO ALEGRE'
 'jardim das flores' 'Cidade nova' 'cidade nova']

Formas de pagamento únicas cadastradas:
['Cartão' 'Dinheiro' 'pix' 'Pix' 'dinheiro' 'cartão' 'cartao' 'Crédito'
 'PIX']


### Tarefa 1.2 — Relatório de problemas

Com base no que você encontrou, preencha a tabela abaixo com **pelo menos 6 problemas de qualidade**, indicando a coluna, a **evidência** (o que você viu na saída do código) e qual das **6 dimensões da qualidade** (Aula 2, seção 0) ele afeta: completude, unicidade, validade, consistência, acurácia ou atualidade.

> ⚠️ Nem todo valor ausente é um problema! Se encontrar algum ausente que **faz sentido**, cite-o e explique por quê.

✍️ **Sua resposta:**

| # | Coluna | Problema | Evidência | Dimensão |
|---|---|---|---|---|
| 1 | tempo_de_entrega | Nulos disfarçados | '?' | 8 |
| 2 | avaliacao | Nulos | NaN | 794 |
| 3 | tempo_de_entrega | Nulos | NaN | 45 |
| 4 | distancia_km | Nulos | NaN | 40 |
| 5 | forma_de_pagamento | Valores diferentes | 'pix', 'PIX' etc | 5 |
| 6 | bairro | Valores diferentes | 'V. Nova', 'vila nova' etc | 34 |

O valor tempo de entrega pode ser zero porque pode ser uma retirada no balcão. Mesma coisa da distância em Km.

A avaliação é um pouco mais complexa. Provavelmente vai ter que fazer uma categoria 'Sem Avaliação' para satisfazer a questão dada.


---

# Parte 2 — Limpeza e qualidade (30 pontos)

📚 *Revise: Aula 2, seções 2 (tipos e formatos), 3 (texto e categorias), 4 (duplicatas) e 5 (valores impossíveis).*

Nesta parte, você vai tratar as **duplicatas** e **três colunas**: `tempo_entrega_min`, `bairro` e `data_hora`. São as colunas de que você vai precisar para responder às perguntas da Marina.

Para **cada** coluna, o seu trabalho deve mostrar:
1. **Antes:** o problema, com código (`value_counts`, formatos, exemplos...);
2. **Tratamento:** o código da limpeza;
3. **Auditoria:** o que mudou — e, se você usou `errors="coerce"`, **quais valores viraram NaN**;
4. **Justificativa:** um texto curto explicando as suas **decisões**.

> As demais colunas (`valor_pedido`, `forma_pagamento`, `distancia_km`...) **não** precisam ser tratadas, a não ser que você queira (não vale ponto extra, mas também não tira).

### Tarefa 2.0 — Cópia de trabalho e duplicatas (3 pontos)

Crie `df` como uma **cópia** de `bruto` e remova as **linhas exatamente duplicadas**. Mostre quantas linhas foram removidas e confirme que não sobrou nenhuma duplicata nem `id_pedido` repetido.

In [10]:
df = bruto.copy()
df2 = df.copy()

# Seu código aqui 👇
df = df.drop_duplicates(keep='first')

print("Linhas removidas: ", bruto.shape[0] - df.shape[0])
print("Linhas duplicadas restantes: ", df.duplicated().sum())
print("IDs repetidos restantes: ", df.duplicated(subset=['id_pedido']).sum())


Linhas removidas:  60
Linhas duplicadas restantes:  0
IDs repetidos restantes:  0


### Tarefa 2.1 — `tempo_entrega_min` (10 pontos)

Esta é a coluna mais importante do projeto: ela aparece em três das quatro perguntas da Marina.

**O que fazer:**
1. Mostre os **formatos** que aparecem na coluna (dica: Aula 2, seção 2.2 — troque dígitos por `9`);
2. Converta a coluna para **número**, tratando os textos que atrapalham a conversão;
3. **Audite** a conversão: liste os valores que viraram `NaN`;
4. Identifique os **valores impossíveis** (use uma regra de domínio: uma entrega real leva **entre 5 e 180 minutos**) e investigue-os: parecem **erros de digitação** com um padrão reconhecível? Decida, para cada tipo de problema, se vai **corrigir** ou transformar em **`NaN`**;
5. Mostre o `describe()` **antes** e **depois**.

> ⚖️ **Não impute** o tempo de entrega com média/mediana: ele é a variável que vamos analisar (Aula 2, seção 6.4). Valores que não puderem ser corrigidos devem ficar como `NaN`.

In [ ]:
# Seu código aqui 👇
#tempoentrega99 = df["tempo_entrega_min"].str.replace(r"\d", "9", regex=True)
#tempoentrega99.value_counts().head(10)

df['tempo_entrega_min'] = df['tempo_entrega_min'].astype(str).str.replace(' min', '', regex=False).str.strip()
df['tempo_entrega_min'] = pd.to_numeric(df['tempo_entrega_min'], errors='coerce')

# Converter negativo para positivo
df['tempo_entrega_min'] = df['tempo_entrega_min'].abs()

valores_fora_do_limite = df[(df['tempo_entrega_min'] < 5) | (df['tempo_entrega_min'] > 180)]
print(valores_fora_do_limite)



      id_pedido            data_hora               bairro  distancia_km  \
88       501181  2025-04-10 21:40:00     Parque das Águas           5.8   
100      501724     29/04/2025 19:13            Vila Nova           3.2   
102      501618  2025-04-25 22:54:00  Distrito Industrial           8.2   
247      500482  2025-03-17 21:32:00            Vila nova           3.8   
345      502533  2025-05-29 13:08:00               Bosque           4.8   
699      502242  2025-05-19 12:03:00     PARQUE DAS AGUAS           6.3   
707      501431  2025-04-19 19:40:00          Santa Luzia           1.5   
838      500870  2025-03-30 12:56:00    Jardim das Flores           2.3   
840      501456  2025-04-20 13:39:00    Jardim das Flores           2.6   
891      501798  2025-05-02 12:54:00    Jardim Das Flores           3.7   
1119     500907  2025-03-31 19:30:00          Alto Alegre           3.7   
1353     501888  2025-05-05 20:03:00     PARQUE DAS AGUAS           7.3   
1473     500977     03/04

In [25]:
# Seu código aqui 👇
# Mostra as primeiras 10 linhas apenas com os prazos e avaliações
maiores_tempos = df[['id_pedido', 'tempo_prometido_min', 'tempo_entrega_min', 'avaliacao']].sort_values(by='tempo_entrega_min', ascending=False)

print("Os 10 maiores tempos de entrega e suas avaliações:")
print(maiores_tempos.head(10))


Os 10 maiores tempos de entrega e suas avaliações:
      id_pedido  tempo_prometido_min  tempo_entrega_min  avaliacao
1353     501888                   50              440.0        NaN
699      502242                   45              430.0        5.0
2559     501760                   40              430.0        4.0
345      502533                   40              420.0        4.0
100      501724                   40              410.0        5.0
88       501181                   40              360.0        4.0
891      501798                   40              250.0        4.0
1796     500544                   25              240.0        5.0
840      501456                   25              230.0        5.0
1473     500977                   35              210.0        5.0


Vemos que não tem sentido um pedido de 430 minutos (mais de 7 horas) ter ganhado uma avaliação de 5 estrelas. É provável que tenha um zero a mais. Por isso, vamos mudar os valores.

In [35]:
# Seu código aqui 👇
df.loc[df['tempo_entrega_min'] > 180, 'tempo_entrega_min'] = df.loc[df['tempo_entrega_min'] > 180, 'tempo_entrega_min'] / 10

df.loc[(df['tempo_entrega_min'] < 5) | (df['tempo_entrega_min'] > 180), 'tempo_entrega_min'] = np.nan

# Auditoria:
maiores_tempos_corrigidos = df[['id_pedido', 'tempo_prometido_min', 'tempo_entrega_min', 'avaliacao']].sort_values(by='tempo_entrega_min', ascending=False)
print("Maiores tempos de entrega:")
print(maiores_tempos_corrigidos.head(20))

print(df(df['tempo_entrega_min'].isna.sum()))

Maiores tempos de entrega:
      id_pedido  tempo_prometido_min  tempo_entrega_min  avaliacao
361      501911                   65               74.0        NaN
1544     500910                   60               71.0        4.0
807      502470                   60               70.0        NaN
1287     500867                   55               69.0        4.0
2110     501164                   65               69.0        3.0
530      501138                   65               69.0        4.0
2640     501017                   60               68.0        4.0
889      502449                   60               65.0        4.0
954      500947                   50               64.0        2.0
522      502348                   50               64.0        NaN
1714     502258                   50               64.0        2.0
886      502301                   50               63.0        4.0
2356     500529                   45               63.0        3.0
1035     500719                   5

AttributeError: 'function' object has no attribute 'sum'

✍️ **Sua resposta — justificativa das decisões sobre `tempo_entrega_min`:**

Os dez maiores tempos de entrega (440 minutos a 210 minutos), não faziam sentido, já que receberam notas de avaliação muito altas. Provavalmente colocaram um zero a mais. Dividi por 10 para fazer sentido. Os outros eu deixei como NaN, provavelmente porque os pedidos foram retirados do balcão.

### Tarefa 2.2 — `bairro` (8 pontos)

Vila Serena tem **exatamente 10 bairros** atendidos (veja o dicionário de dados).

**O que fazer:**
1. Mostre quantos valores diferentes a coluna tem **antes** da limpeza;
2. Padronize os nomes: espaços, maiúsculas/minúsculas, acentos **e** abreviações;
3. **Audite:** mostre se sobrou algum valor fora da lista oficial;
4. Mostre a contagem de pedidos por bairro **antes** e **depois** (um gráfico lado a lado, como na Aula 2, é bem-vindo);
5. No final, os nomes devem ficar **exatamente** como na lista oficial (com acentos e maiúsculas: `"Parque das Águas"`, não `"parque das aguas"`).

In [ ]:
# Seu código aqui 👇

In [ ]:
# Seu código aqui 👇

✍️ **Sua resposta — justificativa das decisões sobre `bairro`:**

### Tarefa 2.3 — `data_hora` (9 pontos)

**O que fazer:**
1. Identifique os **formatos** de data/hora presentes na coluna;
2. Converta para o tipo **datetime**. ⚠️ Lembre-se da armadilha do `format="mixed"` com `dayfirst` (Aula 2, seção 2.3): converta **cada formato explicitamente**;
3. **Audite:** liste os valores que não puderam ser convertidos e explique por que são inválidos;
4. Crie uma nova coluna **`hora`** (número inteiro de 0 a 23) com a hora do pedido — ela será usada na Parte 4;
5. Confira se todas as datas estão dentro do período da exportação (março a maio de 2025).

In [ ]:
# Seu código aqui 👇

In [ ]:
# Seu código aqui 👇

✍️ **Sua resposta — justificativa das decisões sobre `data_hora`:**

### ✅ Validação automática da Parte 2

Execute a célula abaixo **sem alterá-la**. Ela confere se o seu `df` está pronto para as próximas partes. Se algum item aparecer com ❌, volte e corrija a limpeza antes de seguir.

In [ ]:
# ⚠️ NÃO ALTERE ESTA CÉLULA
BAIRROS_OFICIAIS = {"Centro", "Vila Nova", "Jardim das Flores", "Santa Luzia", "Bosque", "Alto Alegre",
                    "Morada do Sol", "Parque das Águas", "Cidade Nova", "Distrito Industrial"}
checagens = {}
checagens["Sem linhas duplicadas"] = df.duplicated().sum() == 0
checagens["Sem id_pedido repetido"] = df["id_pedido"].duplicated().sum() == 0
checagens["Não removeu linhas demais (≥ 2.500 linhas)"] = len(df) >= 2500
checagens["tempo_entrega_min é numérico"] = pd.api.types.is_numeric_dtype(df["tempo_entrega_min"])
if checagens["tempo_entrega_min é numérico"]:
    t = df["tempo_entrega_min"].dropna()
    checagens["tempo_entrega_min entre 5 e 180 (ou NaN)"] = bool(t.between(5, 180).all())
    checagens["tempo_entrega_min com pelo menos 95% preenchido"] = df["tempo_entrega_min"].notna().mean() >= 0.95
checagens["bairro só tem os 10 nomes oficiais"] = set(df["bairro"].dropna().unique()) == BAIRROS_OFICIAIS
checagens["data_hora é datetime"] = pd.api.types.is_datetime64_any_dtype(df["data_hora"])
if checagens["data_hora é datetime"]:
    d = df["data_hora"].dropna()
    checagens["data_hora entre 01/03/2025 e 31/05/2025"] = bool(d.between("2025-03-01", "2025-06-01").all())
checagens["coluna hora existe e vai de 0 a 23"] = ("hora" in df.columns) and bool(df["hora"].dropna().between(0, 23).all())

for nome, ok in checagens.items():
    print(("✅ " if ok else "❌ ") + nome)
print("\n🎉 Parte 2 validada!" if all(checagens.values()) else "\n⚠️ Ainda há itens a corrigir.")

---

# Parte 3 — Respondendo com estatística (30 pontos)

📚 *Revise: Aula 3, seções 2 (descritiva), 5 (intervalos de confiança), 6 (testes de hipótese, tamanho do efeito, como relatar resultados) e 6.7 (qual teste usar).*

Use o `df` limpo da Parte 2. Para cada pergunta, siga **todos** os passos pedidos.

## Pergunta 1 — *"Os pedidos demoram mais em dias de chuva? Quanto?"* (15 pontos)

**a)** Monte uma tabela com **n, média, mediana e desvio padrão** do `tempo_entrega_min` para dias **com** e **sem** chuva.

In [ ]:
# Seu código aqui 👇

**b)** Faça **um gráfico** que compare as **distribuições** do tempo de entrega nos dois grupos (boxplot, histogramas sobrepostos ou ECDF).

In [ ]:
# Seu código aqui 👇

**c)** Escreva as hipóteses $H_0$ e $H_1$ e o nível de significância $\alpha$ que você vai usar.

✍️ **Sua resposta:**

- $H_0$:
- $H_1$:
- $\alpha$ =

**d)** Escolha e aplique **um teste de hipótese adequado**. **Justifique** a escolha (pense em: quantos grupos? que tipo de variável? as variâncias são parecidas? a amostra é grande?). Se quiser, confirme com um segundo teste (por exemplo, a versão não paramétrica).

**e)** Calcule o **intervalo de 95% de confiança da diferença** entre as médias (com chuva − sem chuva).

**f)** Calcule o **tamanho do efeito** (d de Cohen) e classifique-o (pequeno, médio, grande).

In [ ]:
# Seu código aqui 👇

In [ ]:
# Seu código aqui 👇

**g)** Escreva a **conclusão para a Marina**, em 3 a 5 linhas, **sem jargão**. Ela precisa saber: *existe diferença? de quantos minutos, aproximadamente? com que segurança podemos afirmar isso?*

✍️ **Sua resposta:**

## Pergunta 2 — *"A nota que os clientes dão depende do tipo de restaurante?"* (15 pontos)

**a)** A coluna `avaliacao` tem muitos valores ausentes. Calcule a % de ausentes e explique, em 1–2 frases, **o que eles significam** e **o que você vai fazer** com eles nesta análise (lembre-se da Aula 2: nem todo ausente deve ser imputado).

In [ ]:
# Seu código aqui 👇

✍️ **Sua resposta:**

**b)** Para cada `categoria`, mostre: o número de avaliações, a média, a mediana e a **% de cada nota** (1 a 5). *Dica: `pd.crosstab(..., normalize="index")`.* Faça também um gráfico que ajude a comparar as categorias.

In [ ]:
# Seu código aqui 👇

In [ ]:
# Seu código aqui 👇

**c)** Escolha e aplique o **teste adequado** para comparar as **5 categorias**. **Justifique**: a avaliação é uma variável de que tipo (Aula 3, seção 1)? Quantos grupos estão sendo comparados? Escreva $H_0$ e $H_1$.

In [ ]:
# Seu código aqui 👇

✍️ **Sua resposta (hipóteses e justificativa do teste):**

**d)** Escreva a **conclusão para a Marina** (3 a 5 linhas, sem jargão). Atenção à forma de relatar o resultado (Aula 3, seção 6.5): o que você pode — e o que **não** pode — afirmar?

✍️ **Sua resposta:**

---

# Parte 4 — Respondendo com visualização (20 pontos)

📚 *Revise: Aula 4, seções 1 (escolha do gráfico), 3 (cor), 4 (faxina visual e `estilo_storytelling`) e 5 (storytelling).*

Agora os gráficos são **explicativos**: vão ser mostrados para a Marina. Cada gráfico final deve ter:

- ✅ o **tipo de gráfico** adequado à pergunta;
- ✅ um **título-ação** (que diz a conclusão), de preferência com números **calculados**;
- ✅ **faxina visual** (sem bordas, grades e legendas desnecessárias; rótulos diretos quando fizer sentido);
- ✅ **cor com propósito** (cinza + destaque para a mensagem);
- ✅ eixos com **unidade** e a **fonte** dos dados no rodapé.

> Você pode usar matplotlib/seaborn **ou** plotly. Pode fazer gráficos exploratórios "rascunho" antes, mas deixe claro qual é o **gráfico final**.

## Pergunta 3 — *"Em quais horários do dia as entregas são mais lentas?"* (10 pontos)

**a)** Calcule o **tempo de entrega típico** (justifique: média ou mediana?) e o **número de pedidos** para cada `hora` do dia.

In [ ]:
# Seu código aqui 👇

**b)** Faça o **gráfico explicativo** que responde à pergunta, destacando os horários críticos.

In [ ]:
# Seu código aqui 👇

**c)** Interprete em 2–3 frases e sugira **uma ação** para a Marina. *(Dica: compare o gráfico do tempo com o número de pedidos por hora. O que pode explicar os horários mais lentos?)*

✍️ **Sua resposta:**

## Pergunta 4 — *"Quais bairros têm as entregas mais demoradas? É só por causa da distância?"* (10 pontos)

**a)** Faça um **gráfico explicativo** com o **tempo de entrega típico por bairro**, ordenado, deixando claro quais são os bairros mais demorados.

In [ ]:
# Seu código aqui 👇

**b)** Agora investigue se a **distância** explica as diferenças entre os bairros. Faça um **segundo gráfico** relacionando distância e tempo de entrega (por exemplo: a distância típica × o tempo típico de cada bairro, com uma reta de tendência) e **destaque** o(s) bairro(s) que fogem do padrão.

> 💡 **Dica:** dividir o tempo pela distância ("minutos por km") **engana**: parte do tempo (preparo do pedido, saída do restaurante) **não depende** da distância, e isso faz os bairros próximos parecerem "lentos". Prefira comparar cada bairro com a **tendência geral** (a reta) — Aula 3, seção 7, e Aula 4, gráfico G2.

In [ ]:
# Seu código aqui 👇

**c)** Interprete em 2–4 frases: a distância explica a demora? Algum bairro demora **mais do que a distância justificaria**? Quanto, aproximadamente? O que a Marina deveria investigar?

✍️ **Sua resposta:**

---

# ⭐ Bônus (+10 pontos) — *"Atrasar a entrega derruba a nota?"*

1. Crie a coluna `atraso_min` = `tempo_entrega_min` − `tempo_prometido_min` (positivo = chegou **depois** do prometido);
2. Qual a % de pedidos **atrasados**?
3. Agrupe os pedidos em faixas de atraso (por exemplo: *no prazo*, *1–5 min*, *6–10 min*, *mais de 10 min*) e mostre a **avaliação média** e o **n** de cada faixa;
4. Faça um **gráfico explicativo** com a conclusão e meça a relação com uma estatística adequada (por exemplo, a correlação de **Spearman** entre atraso e avaliação, com o p-valor);
5. Conclua para a Marina em 2–3 frases.

In [ ]:
# Seu código aqui 👇

In [ ]:
# Seu código aqui 👇

✍️ **Sua resposta:**

---

## 📝 Resumo para a Marina

Escreva um **resumo executivo** de 4 a 6 linhas respondendo, em linguagem simples, às perguntas da Marina e dizendo **o que você recomenda** que ela faça.

✍️ **Sua resposta:**

---

## ✅ Checklist antes de enviar

- [ ] Preenchi os nomes na identificação;
- [ ] Todas as células de código e de texto estão preenchidas;
- [ ] A validação da Parte 2 mostra 🎉;
- [ ] Executei **Restart & Run All** e o notebook rodou até o fim **sem erros**;
- [ ] Salvei o notebook **com as saídas** visíveis;
- [ ] Renomeei o arquivo para `projeto_eda_<nome1>_<nome2>.ipynb`;
- [ ] Enviei para **jchambyd@gmail.com** com o assunto `Projeto EDA – <Nome 1> / <Nome 2>` até **20/10/2026**.

**Bom trabalho! 🚀**